In [1]:
import sys
import cv2
from pathlib import Path

In [2]:
FACE_LANDMARKER_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)

def ensure_face_landmarker_model(model_path: str = "face_landmarker.task") -> str:
    path = Path(model_path)
    if path.exists() and path.stat().st_size > 100_000:
        return str(path)
    
    from urllib.request import urlretrieve
    
    print(f"Descargando modelo MediaPipe FaceLandmarker en {path}...")
    urlretrieve(FACE_LANDMARKER_URL, str(path))
    print(f"Modelo descargado ({path.stat().st_size / 1e6:.2f} MB).")
    
    return str(path)

In [3]:
def open_camera_and_options(camera_index: int):
    # if linux, use V4L2 backend for better performance
    cap = None
    if sys.platform.startswith("linux"):
        cap = cv2.VideoCapture(camera_index, cv2.CAP_V4L2)
        cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
    elif sys.platform == "darwin":
        cap = cv2.VideoCapture(camera_index)
    elif sys.platform == "win32":
        cap = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)
    else:
        cap = cv2.VideoCapture(camera_index)
    
    if not cap.isOpened():
        print(f"No se pudo abrir la cámara {camera_index}. Verifica la conexión o usa otro índice.")
        return None
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    
    return cap

In [4]:
from ironmanhudv2.biometry_engine.engine_config import BiometryConfig, ParamPose3D, ParamEAR, ParamMAR

EAR_PARAMS = ParamEAR(
    threshold=0.21,
    consec_frames_blink=2,
    micro_sleep_seconds=1.2,
    exp_moving_avg_alpha=0.2,
)

MAR_PARAMS = ParamMAR(
    threshold=0.55,
    yawn_seconds=1.0,
    exp_moving_avg_alpha=0.45,
)

POSE3D_PARAMS = ParamPose3D(
    pitch_nod_threshold=-18.0,
    pitch_up_threshold=25.0,
    yaw_look_away_threshold=28.0,
    pose_alert_seconds=1.0,
    exp_moving_avg_alpha=0.4
)

BIOMETRY_CONFIG = BiometryConfig(
    model_path=ensure_face_landmarker_model(),
    emotion_model_path=Path("outputs_p5/models/v1_CNN_torch/cnn_fer_torch_seed42.pt").resolve().as_posix(),
    params_EAR=EAR_PARAMS,
    params_MAR=MAR_PARAMS,
    params_pose3D=POSE3D_PARAMS,
    alarm_cooldown_seconds=1.8,
    history_len=75
)

In [ ]:
from ironmanhudv2.renderer import IronManHUDRenderer

def run_ironman_hud(camera_index = 0):
    cap = open_camera_and_options(camera_index,)
    if cap is None:
        return

    ironman_hud_renderer = IronManHUDRenderer()
    ironman_hud_renderer.setup(BIOMETRY_CONFIG, False)
    window_name = "Ironman HUD"
    cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
    window_visible_seen = False

    print("Iniciando Ironman HUD. Presiona 'c' para calibrar la pose o 'q'/ESC para salir.")
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("No se pudo leer el cuadro de la cámara.")
                break

            hud_frame = ironman_hud_renderer.render(frame)
            cv2.imshow(window_name, hud_frame)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord('q'), ord('Q'), 27):
                break
            elif key in (ord('c'), ord('C')):
                offset = ironman_hud_renderer.register_calibration()
                if offset is None:
                    print("No se detectó rostro; no se pudo calibrar la pose.")
                else:
                    print(f"Pose calibrada (P={offset[0]:+.1f}°, Y={offset[1]:+.1f}°, R={offset[2]:+.1f}°)")

            visible = cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE)
            if visible >= 1:
                window_visible_seen = True
            elif window_visible_seen:
                break
    except KeyboardInterrupt:
        print("Ejecución interrumpida por el usuario.")
    finally:
        cap.release()
        ironman_hud_renderer.close()
        cv2.destroyAllWindows()
        for _ in range(5):
            cv2.waitKey(1)
    
run_ironman_hud()

/home/skinnydevi/Development/computer-vision/P5/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
W0000 00:00:1795725928.412255   74959 face_landmarker_graph.cc:181] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795725928.422498   74964 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1795725928.431463   74964 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling su

Iniciando Ironman HUD. Presiona 'c' para calibrar la pose o 'q'/ESC para salir.
Pose calibrada (P=-1.1°, Y=+13.3°, R=-0.6°)
